In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import pmdarima as pm
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DATA_ROOT  = "data/resampled-v2"
SPLITS     = [
    ("2024-06-01 (80/20)", "ratio"),
    ("2024-10-01",          "2024-10-01"),
    ("2025-01-01",          "2025-01-01"),
]
INTERVALS  = ["1min", "10min", "1h", "1d"]
# pandas frequency strings
FREQ_MAP   = {"1min":"T", "10min":"10T", "1h":"H", "1d":"D"}

# auto_arima parallelism and CV folds
N_JOBS   = -1
CV_FOLDS = 3

# suppress all warnings
warnings.filterwarnings("ignore")

results = []

# -----------------------------------------------------------------------------
# Main loop: for each split, each interval, each coin
# -----------------------------------------------------------------------------
for split_label, dirname in SPLITS:
    split_dir = os.path.join(DATA_ROOT, dirname)
    print(f"\n===== SPLIT: {split_label} =====")

    for iv in INTERVALS:
        print(f"\n--- Interval: {iv} ---")

        # load train/test sets
        train_df = pd.read_parquet(os.path.join(split_dir, f"train_{iv}.parquet"))
        test_df  = pd.read_parquet(os.path.join(split_dir, f"test_{iv}.parquet"))
        train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
        test_df ["timestamp"] = pd.to_datetime(test_df["timestamp"])

        for coin in train_df["coin_id"].unique():
            print(f"\nCoin: {coin}")

            # build log-return series
            ts_tr = (
                train_df[train_df.coin_id == coin]
                .set_index("timestamp")["close"]
                .asfreq(FREQ_MAP[iv])
                .dropna()
            )
            lr_tr = np.log(ts_tr).diff().dropna()

            # choose ARIMA order
            if iv == "1min":
                # fixed ARIMA(1,0,1) on 1-minute returns
                p, d, q = 1, 0, 1
                print("  Using fixed ARIMA order (1,0,1) for 1min interval")
            else:
                # automatic order selection for other intervals
                model = pm.auto_arima(
                    lr_tr,
                    start_p=0, start_q=0,
                    max_p=3, max_q=3,
                    seasonal=False,
                    stepwise=True,
                    n_jobs=N_JOBS,
                    error_action="ignore",
                    suppress_warnings=True,
                    information_criterion="aic"
                )
                p, d, q = model.order
                print(f"  Selected ARIMA order=(p,d,q)=({p},{d},{q}), AIC={model.aic():.2f}")

            # 1) cross-validation with fixed order
            tscv = TimeSeriesSplit(n_splits=CV_FOLDS)
            cv_rmses = []
            for fold, (train_idx, val_idx) in enumerate(tscv.split(lr_tr), start=1):
                tr_fold  = lr_tr.iloc[train_idx]
                val_fold = lr_tr.iloc[val_idx]
                m = pm.ARIMA(order=(p, d, q)).fit(
                    tr_fold,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    suppress_warnings=True
                )
                preds = m.predict(n_periods=len(val_fold))
                rmse  = np.sqrt(mean_squared_error(val_fold, preds))
                cv_rmses.append(rmse)
                print(f"    Fold {fold} RMSE={rmse:.4f}")
            cv_mean, cv_std = np.mean(cv_rmses), np.std(cv_rmses)
            print(f"  CV RMSE = {cv_mean:.4f} ± {cv_std:.4f}")

            # 2) final fit on full train set
            final_model = pm.ARIMA(order=(p, d, q)).fit(
                lr_tr,
                enforce_stationarity=False,
                enforce_invertibility=False,
                suppress_warnings=True
            )

            # prepare test log-returns
            ts_te = (
                test_df[test_df.coin_id == coin]
                .set_index("timestamp")["close"]
                .asfreq(FREQ_MAP[iv])
                .dropna()
            )
            lr_te = np.log(ts_te).diff().dropna()

            # 3) forecast on test
            preds_test = final_model.predict(n_periods=len(lr_te))
            rmse_test  = np.sqrt(mean_squared_error(lr_te, preds_test))
            print(f"  Test RMSE={rmse_test:.4f}")

            # record results
            results.append({
                "split":      split_label,
                "interval":   iv,
                "coin":       coin,
                "p":          p,
                "d":          d,
                "q":          q,
                "aic":        final_model.aic(),
                "cv_rmse":    cv_mean,
                "cv_std":     cv_std,
                "test_rmse":  rmse_test
            })

# -----------------------------------------------------------------------------
# Summarize & save
# -----------------------------------------------------------------------------
df_res = pd.DataFrame(results)
print("\n=== Summary ===")
print(df_res)

os.makedirs("data", exist_ok=True)
df_res.to_csv("data/ARIMA_baseline_summary.csv", index=False)


===== SPLIT: 2024-06-01 (80/20) =====

--- Interval: 1min ---

Coin: BNBUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0011
    Fold 2 RMSE=0.0008
    Fold 3 RMSE=0.0008
  CV RMSE = 0.0009 ± 0.0001
  Test RMSE=0.0008

Coin: BTCUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0010
    Fold 2 RMSE=0.0007
    Fold 3 RMSE=0.0007
  CV RMSE = 0.0008 ± 0.0001
  Test RMSE=0.0007

Coin: ETHUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0012
    Fold 2 RMSE=0.0008
    Fold 3 RMSE=0.0008
  CV RMSE = 0.0009 ± 0.0002
  Test RMSE=0.0010

Coin: SOLUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0017
    Fold 2 RMSE=0.0016
    Fold 3 RMSE=0.0015
  CV RMSE = 0.0016 ± 0.0001
  Test RMSE=0.0013

Coin: XRPUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0013
    Fold 2 RMSE=0.0012
    Fold 3 RMSE=0.0010
  CV RMSE = 0.0012 ± 0.0001
  Test RMSE=0.0015

--- Interval: 10min -

## Data Prep for P&L backtest

In [3]:
import os
import warnings

import numpy as np
import pandas as pd
import pmdarima as pm
from scipy.stats import mstats

# -----------------------------------------------------------------------------
# 0) suppress warnings
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1) Configuration
# -----------------------------------------------------------------------------
DATA_ROOT = "data/resampled-v2"
RESULTS_ROOT = "results/ARIMA"
SPLITS = [
    ("2024-06-01 (80/20)", "ratio"),
    ("2024-10-01",          "2024-10-01"),
    ("2025-01-01",          "2025-01-01"),
]
INTERVALS = ["1min", "10min", "1h", "1d"]
# frequency map for pandas asfreq
FREQ_MAP = {"1min": "min", "10min": "10min", "1h": "1h", "1d": "1d"}

# -----------------------------------------------------------------------------
# 2) load best (p,d,q) per coin
# -----------------------------------------------------------------------------
params_df = pd.read_csv("data/ARIMA_baseline_summary.csv")

# -----------------------------------------------------------------------------
# 3) generate and save per‐coin backtest inputs
# -----------------------------------------------------------------------------
for split_label, dirname in SPLITS:
    split_dir = os.path.join(DATA_ROOT, dirname)
    for iv in INTERVALS:
        # prepare output folder for this split & interval
        out_dir = os.path.join(RESULTS_ROOT, dirname, iv)
        os.makedirs(out_dir, exist_ok=True)

        # load train/test
        train_df = pd.read_parquet(os.path.join(split_dir, f"train_{iv}.parquet"))
        test_df  = pd.read_parquet(os.path.join(split_dir, f"test_{iv}.parquet"))
        train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
        test_df ["timestamp"] = pd.to_datetime(test_df["timestamp"])

        # select coins with parameters for this split/interval
        cond = (params_df["split"] == split_label) & (params_df["interval"] == iv)
        coins = params_df.loc[cond, "coin"].unique()

        for coin in coins:
            # get best order
            row = params_df[cond & (params_df["coin"] == coin)].iloc[0]
            p, d, q = int(row.p), int(row.d), int(row.q)

            # training log-returns
            ts_tr = (
                train_df[train_df.coin_id == coin]
                .set_index("timestamp")["close"]
                .asfreq(FREQ_MAP[iv])
                .dropna()
            )
            lr_tr = np.log(ts_tr).diff().dropna()

            # fit ARIMA
            model = pm.ARIMA(order=(p, d, q)).fit(
                lr_tr,
                enforce_stationarity=False,
                enforce_invertibility=False,
                suppress_warnings=True
            )

            # test log-returns and bar returns
            ts_te = (
                test_df[test_df.coin_id == coin]
                .set_index("timestamp")["close"]
                .asfreq(FREQ_MAP[iv])
                .dropna()
            )
            lr_te = np.log(ts_te).diff().dropna()
            bar_ret = ts_te.pct_change().shift(-1).loc[lr_te.index]

            # forecast log-returns
            preds = model.predict(n_periods=len(lr_te))

            # assemble DataFrame for backtesting
            df_bt = pd.DataFrame({
                "pred":    preds,
                "signal":  np.sign(preds),
                "bar_ret": bar_ret
            }, index=lr_te.index)

            # filter extreme jumps
            df_bt = df_bt[df_bt["bar_ret"].abs() <= 0.20]

            # save for later backtest
            df_bt.to_parquet(os.path.join(out_dir, f"{coin}.parquet"))